# Lab 04 – Webdev 101, a Web Page for Your Chart

| | |
|---|---|
| **Name** | Usman Hassan |
| **Roll number** | 1421-324032 |
| **Section** | *(write your section here)* |
| **Date** | 16 November 2026 |
| **AI tools used** | Claude (Anthropic) – helped write and check the code; I can explain every line. |

In [1]:
# the data file was uploaded to Colab (folder icon > Upload)
import os
print("students_marks.csv", "found" if os.path.exists("students_marks.csv") else "MISSING")

students_marks.csv found


## Task 1 – An HTML page from Python

In [2]:
import csv
from IPython.display import HTML, display

# 2. loading the marks file: every row becomes a dictionary, every value is text
with open("students_marks.csv", encoding="utf-8") as f:
    students = list(csv.DictReader(f))

subjects = ["Math", "Stats", "Python", "Databases", "English"]

# converting the five marks to numbers and adding the Average key (1 decimal)
for s in students:
    for sub in subjects:
        s[sub] = int(s[sub])
    s["Average"] = round(sum(s[sub] for sub in subjects) / len(subjects), 1)

print(len(students), "students loaded")
print(students[0])

25 students loaded
{'RollNo': 'BSDS-01', 'Name': 'Ali Khan', 'Section': 'B', 'Math': 66, 'Stats': 80, 'Python': 81, 'Databases': 74, 'English': 69, 'Average': 74.0}


In [3]:
# 3. top 5 students by Average, turned into HTML table rows
top5 = sorted(students, key=lambda s: s["Average"], reverse=True)[:5]
rows = ""
for s in top5:
    rows += (f"<tr><td>{s['RollNo']}</td><td>{s['Name']}</td>"
             f"<td>{s['Section']}</td><td>{s['Average']}</td></tr>\n      ")

# link to my semester project dataset
project_url = "https://archive.ics.uci.edu/dataset/352/online+retail"

results_page = f"""<!DOCTYPE html>
<meta charset="utf-8">
<body>
  <h1>BSDS Results Dashboard</h1>
  <p>Made from <span class="file">students_marks.csv</span>,
     the marks of {len(students)} students in five subjects.</p>
  <h2>What the dashboard will show</h2>
  <ul>
    <li>A bar chart of every student's average, highest first</li>
    <li>Key numbers: students, class average, number passed and the topper</li>
    <li>The students below 60, marked in red</li>
  </ul>
  <p>My semester project dataset:
     <a href="{project_url}">Online Retail (UCI Machine Learning Repository)</a></p>
  <table id="top5" border="1" cellpadding="4">
    <caption>Top 5 students by Average</caption>
    <tr><th>RollNo</th><th>Name</th><th>Section</th><th>Average</th></tr>
      {rows}
  </table>
</body>"""

# 4. saving the file, showing it and printing the top 5
with open("results.html", "w", encoding="utf-8") as f:
    f.write(results_page)
display(HTML(results_page))

print("Top 5 by Average (saved in results.html):")
for s in top5:
    print(" ", s["Name"], s["Average"])

RollNo,Name,Section,Average
BSDS-20,Rabia Aslam,B,100.0
BSDS-18,Laiba Khan,C,89.6
BSDS-08,Fatima Noor,A,85.4
BSDS-17,Saad Anwar,A,80.8
BSDS-19,Hassan Butt,A,78.8


Top 5 by Average (saved in results.html):
  Rabia Aslam 100.0
  Laiba Khan 89.6
  Fatima Noor 85.4
  Saad Anwar 80.8
  Hassan Butt 78.8


## Task 2 – CSS classes and ids

In [4]:
# 1. all 25 students sorted by Average, highest first
ranked = sorted(students, key=lambda s: s["Average"], reverse=True)
topper = ranked[0]
low = [s for s in ranked if s["Average"] < 60]

# 2. class low below 60, class ok for the rest, id topper on the first row
rows = ""
for s in ranked:
    row_class = "low" if s["Average"] < 60 else "ok"
    row_id = ' id="topper"' if s is topper else ""
    rows += (f'<tr class="{row_class}"{row_id}><td>{s["Name"]}</td>'
             f'<td>{s["Section"]}</td><td>{s["Average"]}</td></tr>\n')

# 3. the CSS: dark header row, red bold for .low, gold for #topper
css = """
#all-students { border-collapse: collapse; font-family: sans-serif; }
#all-students th, #all-students td { border: 1px solid #999; padding: 3px 12px; }
#all-students th { background: #222; color: white; }
#all-students .low { color: red; font-weight: bold; }
#all-students #topper { background: gold; }
"""
table = f"""<table id="all-students">
<tr><th>Name</th><th>Section</th><th>Average</th></tr>
{rows}</table>"""
display(HTML("<style>" + css + "</style>" + table))

# 4. the counts
print(len(ranked), "rows in the table")
print(len(low), "students are below 60:", ", ".join(s["Name"] for s in low))
print("Topper:", topper["Name"], "with an average of", topper["Average"])

Name,Section,Average
Rabia Aslam,B,100.0
Laiba Khan,C,89.6
Fatima Noor,A,85.4
Saad Anwar,A,80.8
Hassan Butt,A,78.8
Noor Fatima,B,77.6
Danish Iqbal,A,77.4
Omar Farooq,A,76.6
Ali Khan,B,74.0
Sana Rauf,A,70.8


25 rows in the table
5 students are below 60: Mahnoor Zafar, Hina Malik, Hamza Raza, Faisal Qureshi, Asad Mirza
Topper: Rabia Aslam with an average of 100.0


## Task 3 – The placeholder dashboard

In [5]:
import os

# 3. the four sidebar numbers, computed in Python
n_students = len(students)
class_avg = round(sum(s["Average"] for s in students) / n_students, 1)
n_passed = len([s for s in students if s["Average"] >= 50])
topper_name = topper["Name"]

os.makedirs("dashboard", exist_ok=True)

# 1-2. index.html: header, main box and sidebar inside the chart holder
html = f"""<!DOCTYPE html>
<meta charset="utf-8">
<link rel="stylesheet" href="style.css">
<body>
  <div id="chart-holder" class="dev">
    <div id="header">
      <h2>BSDS Results Dashboard</h2>
      <p>Average marks of {n_students} BSDS students in Math, Stats, Python,
         Databases and English, from students_marks.csv.</p>
    </div>
    <div id="chart-components">
      <div id="main">A placeholder for the main chart.</div>
      <div id="sidebar">
        <p>Students<br><span class="number">{n_students}</span></p>
        <p>Class average<br><span class="number">{class_avg}</span></p>
        <p>Passed (50 or more)<br><span class="number">{n_passed}</span></p>
        <p>Topper<br><span class="number">{topper_name}</span></p>
        <p id="note"></p>
      </div>
    </div>
  </div>
  <script src="script.js"></script>
</body>"""

# 4. style.css: page 720 px wide, chart area 320 px high, dev borders kept
css = """
body { background: #ccc; font-family: sans-serif; }
div.dev { border: solid 1px red; }
div.dev div { border: dashed 1px green; }
div#chart-holder { width: 720px; background: white; margin: auto; }
div#header { padding: 0 10px; }
div#chart-components { height: 320px; position: relative; }
div#main, div#sidebar { position: absolute; }
div#main { width: 75%; height: 100%; background: #eee; }
div#sidebar { right: 0; width: 25%; height: 100%; font-size: 13px; }
div#sidebar p { margin: 8px 10px; }
div#sidebar .number { font-size: 18px; font-weight: bold; color: #0F6E7C; }
div#sidebar #note { font-size: 11px; color: #555; }
"""

# script.js writes a small note in the last line of the sidebar
js = """document.querySelector("#note").textContent =
    "Sidebar filled by Python, this note by script.js";"""

for name, text in [("index.html", html), ("style.css", css), ("script.js", js)]:
    with open("dashboard/" + name, "w", encoding="utf-8") as f:
        f.write(text)

def show(folder):
    """Put style.css and script.js inside the page, then show it in Colab."""
    page = open(folder + "/index.html", encoding="utf-8").read()
    css = open(folder + "/style.css", encoding="utf-8").read()
    js = open(folder + "/script.js", encoding="utf-8").read()
    page = page.replace('<link rel="stylesheet" href="style.css">',
                        "<style>" + css + "</style>")
    page = page.replace('<script src="script.js"></script>',
                        "<script>" + js + "</script>")
    display(HTML(page))

# 5. showing the page and printing the four numbers
show("dashboard")
print(sorted(os.listdir("dashboard")))
print("Students:", n_students)
print("Class average:", class_avg)
print("Passed (Average 50 or more):", n_passed)
print("Topper:", topper_name)

['index.html', 'script.js', 'style.css']
Students: 25
Class average: 70.2
Passed (Average 50 or more): 24
Topper: Rabia Aslam


## Task 4 – A hand-made SVG line chart

In [6]:
# 1. class average of each subject
avg = {sub: sum(s[sub] for s in students) / len(students) for sub in subjects}
print({sub: round(v, 1) for sub, v in avg.items()})

# 3. the scale: mark 40 sits on the x-axis (y = 180)
def y_of(mark):
    return 180 - (mark - 40) * 3.5

# one point per subject: x goes 70 px to the right each time
points = [(60 + i * 70, y_of(avg[sub])) for i, sub in enumerate(subjects)]

# the path: M moves the pen to the first point, L draws a line to the next ones
d = "M" + "L".join(f"{x} {y:.1f}" for x, y in points)

circles = ""
names = ""
for (x, y), sub in zip(points, subjects):
    circles += f'<circle cx="{x}" cy="{y:.1f}" r="4"></circle>\n  '
    names += f'<text class="tick" x="{x}" y="196">{sub}</text>\n  '

# small numbers on the y-axis, using the same scale
ticks = ""
for mark in [40, 60, 80]:
    ticks += f'<text class="ytick" x="34" y="{y_of(mark) + 4:.1f}">{mark}</text>\n  '

# 2. the SVG, styled with CSS in a <style> tag
svg = f"""
<style>
  #line-chart {{ background: #eee; font-family: sans-serif; }}
  #line-chart text {{ font-size: 11px; }}
  #line-chart text#title {{ font-size: 15px; text-anchor: middle; }}
  #line-chart text.tick {{ text-anchor: middle; }}
  #line-chart text.ytick {{ text-anchor: end; }}
  #line-chart line {{ stroke: #555555; stroke-width: 2; }}
  #line-chart path {{ stroke: #0F6E7C; stroke-width: 2; fill: none; }}
  #line-chart circle {{ fill: #0F6E7C; }}
  #line-chart rect {{ stroke: #555555; fill: white; }}
  #line-chart line.key {{ stroke: #0F6E7C; }}
</style>
<svg id="line-chart" width="380" height="210">
  <text id="title" x="190" y="18">Class average per subject</text>
  <line x1="40" y1="30" x2="40" y2="180"></line>
  <line x1="40" y1="180" x2="370" y2="180"></line>
  <text x="40" y="30" transform="rotate(-90,40,30)"
        text-anchor="end" dy="-2.2em">Average marks</text>
  <rect x="262" y="28" width="108" height="20"></rect>
  <line class="key" x1="268" y1="38" x2="288" y2="38"></line>
  <text x="293" y="42">Class average</text>
  <path d="{d}"></path>
  {circles}
  {names}
  {ticks}
</svg>
"""
display(HTML(svg))
print("d =", d)

{'Math': 70.1, 'Stats': 70.1, 'Python': 73.4, 'Databases': 71.4, 'English': 66.3}


d = M60 74.6L130 74.7L200 63.1L270 70.2L340 88.0


## Task 5 – Fill the main box with a bar chart

In [7]:
# 1. students sorted by Average, one bar each, inside a group that keeps a margin
ranked = sorted(students, key=lambda s: s["Average"], reverse=True)

W, H = 520, 300
left, top, right, bottom = 40, 20, 10, 70
inner_w = W - left - right        # 470 px for the 25 bars
inner_h = H - top - bottom        # 210 px = 100 marks
step = inner_w / len(ranked)      # space for one student
bar_w = step - 4

def bar_y(mark):
    return inner_h - mark / 100 * inner_h   # y is measured down from the top

# 2. class low for the bars below 60, first name rotated -60 under each bar
bars = ""
for i, s in enumerate(ranked):
    x = i * step + 2
    y = bar_y(s["Average"])
    bar_class = "bar low" if s["Average"] < 60 else "bar"
    first_name = s["Name"].split()[0]
    cx = x + bar_w / 2
    bars += (f'<rect class="{bar_class}" x="{x:.1f}" y="{y:.1f}" '
             f'width="{bar_w:.1f}" height="{inner_h - y:.1f}"></rect>\n      '
             f'<text class="name" x="{cx:.1f}" y="{inner_h + 10}" '
             f'transform="rotate(-60,{cx:.1f},{inner_h + 10})">{first_name}</text>\n      ')

# numbers on the y-axis
ticks = ""
for mark in [0, 20, 40, 60, 80, 100]:
    ticks += f'<text class="tick" x="-5" y="{bar_y(mark) + 3:.1f}">{mark}</text>\n      '

bar_svg = f"""<svg id="bars" width="{W}" height="{H}">
    <g transform="translate({left}, {top})">
      {bars}
      {ticks}
      <line class="axis" x1="0" y1="0" x2="0" y2="{inner_h}"></line>
      <line class="axis" x1="0" y1="{inner_h}" x2="{inner_w}" y2="{inner_h}"></line>
      <line class="pass" x1="0" y1="{bar_y(60):.1f}" x2="{inner_w}" y2="{bar_y(60):.1f}"></line>
    </g>
  </svg>"""

# 3. the SVG replaces the placeholder text in #main, new rules go into style.css
dashboard_html = html.replace("A placeholder for the main chart.", bar_svg)

bar_css = """
#bars .bar { fill: #0F6E7C; }
#bars .bar.low { fill: red; }
#bars text { font-size: 10px; }
#bars text.name, #bars text.tick { text-anchor: end; }
#bars line.axis { stroke: #555555; }
#bars line.pass { stroke: #555555; stroke-dasharray: 5 4; }
"""

with open("dashboard/index.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)
with open("dashboard/style.css", "w", encoding="utf-8") as f:
    f.write(css + bar_css)

show("dashboard")
print(dashboard_html.count("<rect"), "bars,",
      dashboard_html.count('class="bar low"'), "of them red (below 60)")

25 bars, 5 of them red (below 60)


In [8]:
# 4. zipping the folder and downloading it, to open index.html in Chrome
import shutil
shutil.make_archive("dashboard", "zip", ".", "dashboard")
try:
    from google.colab import files
    files.download("dashboard.zip")
except ImportError:
    print("Not in Colab: dashboard.zip is saved next to the notebook")

Not in Colab: dashboard.zip is saved next to the notebook


**DevTools:** I unzipped the folder, opened `index.html` in Chrome, pressed Ctrl+Shift+I, selected one red bar in the Elements tab and changed its `fill` to another colour. The screenshot is saved as `devtools.png` in this folder.

## Task 6 – A D3 scatter plot

In [9]:
import json

# 1. one dictionary per student: x from Math, y from Python, r from Average
circles = [{"x": (s["Math"] - 30) * 4.5,
            "y": 340 - (s["Python"] - 30) * 4.5,
            "r": s["Average"] / 10,
            "section": s["Section"],
            "name": s["Name"]}
           for s in students]
data = {"width": 360, "height": 380, "circles": circles}
print(len(circles), "circles, first:", circles[0])

# 2-3. chartCircles also gives each circle class = its Section; CSS colours A, B, C
scatter_page = """<!DOCTYPE html>
<meta charset="utf-8">
<style>
  #scatter { background: #eee; font-family: sans-serif; }
  #scatter circle { opacity: 0.6; }
  #scatter circle.A { fill: red; }
  #scatter circle.B { fill: blue; }
  #scatter circle.C { fill: green; }
  #scatter line { stroke: #555555; }
  #scatter text { font-size: 13px; text-anchor: middle; }
  #scatter text.key { font-size: 11px; text-anchor: start; }
</style>
<body>
<svg id="scatter">
  <line x1="40" y1="340" x2="350" y2="340"></line>
  <line x1="40" y1="10" x2="40" y2="340"></line>
  <text x="195" y="365">Math</text>
  <text x="18" y="175" transform="rotate(-90,18,175)">Python</text>
  <circle class="A" cx="270" cy="290" r="5"></circle><text class="key" x="280" y="294">Section A</text>
  <circle class="B" cx="270" cy="306" r="5"></circle><text class="key" x="280" y="310">Section B</text>
  <circle class="C" cx="270" cy="322" r="5"></circle><text class="key" x="280" y="326">Section C</text>
</svg>
<script>
var chartCircles = function(data) {
  var chart = d3.select("#scatter");
  chart.attr("height", data.height).attr("width", data.width);
  chart.selectAll("circle.student").data(data.circles)
    .enter()
    .append("circle")
    .attr("class", function(d) { return "student " + d.section; })
    .attr("cx", function(d) { return d.x; })
    .attr("cy", function(d) { return d.y; })
    .attr("r", function(d) { return d.r; });
};
var data = DATA;
// load D3 first, then draw
var s = document.createElement("script");
s.src = "https://cdn.jsdelivr.net/npm/d3@7";
s.onload = function() { chartCircles(data); };
document.head.appendChild(s);
</script>
</body>
""".replace("DATA", json.dumps(data))

with open("scatter.html", "w", encoding="utf-8") as f:
    f.write(scatter_page)
display(HTML(scatter_page))
print("saved scatter.html")

25 circles, first: {'x': 162.0, 'y': 110.5, 'r': 7.4, 'section': 'B', 'name': 'Ali Khan'}


saved scatter.html


**4. The big circle in the top-right corner** is **Rabia Aslam** (BSDS-20, Section B). She has 100 in Math, so her circle is furthest to the right, and 100 in Python, so it is at the very top. The radius is Average / 10, and her Average is 100 (100 in all five subjects), the highest in the class, so r = 10 is the biggest radius.

## Task 7 – Power BI results page

Built in Power BI Desktop: a text box header "BSDS Results Dashboard", a clustered column chart of Average by Section (aggregation set to Average, not Sum), two cards (Count of RollNo and Average of Average) and a table of Name and Average with the font red below 60.

Files: `lab04_report.pbix` and `lab04_report.pdf` in this folder.

## Task 8 – Upload to GitHub

Uploaded to the Lab 04 folder of `data-viz-1421-324032`: `webdev.ipynb`, the `dashboard` folder (`index.html`, `style.css`, `script.js`), `devtools.png`, `lab04_report.pbix` and `lab04_report.pdf`.